# trainpit tutorial notebook

This notebook demonstrates the current `train` tracker API with a deterministic dummy training loop. Terminal rendering is still under development, so the notebook records a local history list to show the values being sent to trainpit.

In [ ]:
from __future__ import annotations

import math

from trainpit import train

TOTAL_EPOCHS = 3
TOTAL_STEPS = 8


def training_step(epoch: int, step: int) -> tuple[float, float, float]:
    """Return deterministic demo loss, accuracy, and learning rate values."""

    progress_ratio = ((epoch - 1) * TOTAL_STEPS + step) / (TOTAL_EPOCHS * TOTAL_STEPS)
    loss = 1.2 * math.exp(-2.0 * progress_ratio) + 0.02 * math.sin(step)
    accuracy = 0.45 + 0.5 * progress_ratio
    learning_rate = 0.001 * (1.0 - 0.7 * progress_ratio)
    return round(loss, 4), round(accuracy, 4), round(learning_rate, 6)

## Run a training loop

Wrap the loop with `train(...)`, then update epoch, step, loss, metrics, and learning rate as training progresses.

In [ ]:
history: list[dict[str, float | int]] = []

with train(
    total_epochs=TOTAL_EPOCHS, total_steps=TOTAL_STEPS, label="notebook-demo"
) as progress:
    for epoch in range(1, TOTAL_EPOCHS + 1):
        progress.epoch(epoch)

        for step in range(1, TOTAL_STEPS + 1):
            loss, accuracy, learning_rate = training_step(epoch, step)

            progress.step(
                step,
                loss=loss,
                metrics={"acc": accuracy},
                learning_rate=learning_rate,
            )

            history.append(
                {
                    "epoch": epoch,
                    "step": step,
                    "loss": loss,
                    "acc": accuracy,
                    "lr": learning_rate,
                }
            )

    progress.log("notebook demo complete")

assert len(history) == TOTAL_EPOCHS * TOTAL_STEPS
assert float(history[-1]["loss"]) < float(history[0]["loss"])

print(f"Recorded {len(history)} training updates")
print(f"First loss: {history[0]['loss']}")
print(f"Final loss: {history[-1]['loss']}")

## Inspect the values

The current tracker stores update state for renderers. Until rendering lands, you can still use normal Python data structures to inspect or visualize the same values in notebooks.

In [ ]:
print("epoch step loss   acc    lr")
for row in [*history[::4], history[-1]]:
    print(
        f"{int(row['epoch']):>5} {int(row['step']):>4} "
        f"{float(row['loss']):>6.4f} {float(row['acc']):>6.4f} {float(row['lr']):>8.6f}"
    )

## Show a simple learning curve

This is a notebook-only ASCII preview of the learning curve direction. Trainpit's terminal graph renderer is planned separately.

In [ ]:
LEVELS = " .:-=+*#%@"


def sparkline(values: list[float], width: int = 24) -> str:
    if not values:
        return ""

    if len(values) > width:
        indexes = [
            round(index * (len(values) - 1) / (width - 1)) for index in range(width)
        ]
        samples = [values[index] for index in indexes]
    else:
        samples = values

    low = min(samples)
    high = max(samples)
    if high == low:
        return LEVELS[0] * len(samples)

    scale = len(LEVELS) - 1
    return "".join(
        LEVELS[round((value - low) / (high - low) * scale)] for value in samples
    )


losses = [float(row["loss"]) for row in history]
accuracies = [float(row["acc"]) for row in history]

print(f"loss {losses[0]:.4f} -> {losses[-1]:.4f}  {sparkline(losses)}")
print(f"acc  {accuracies[0]:.4f} -> {accuracies[-1]:.4f}  {sparkline(accuracies)}")